In [1]:
from utils import calculate_nse, create_lag, create_lag_adv, create_lag_adv_corrected

In [2]:
#tips
#for single column dataframes always use double brackets, single brackets will make it a series, or float type
# for adding two columns to a df either add them one by one or use two brackets

In [3]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)
df1 = df.copy()
df.head()


,dewpoint,et_loss,ndvi,precipitation,radiation,sca,dd,temp,runoff
date,,,,,,,,,
2000-06-30,270.274111,0.035384,0.167895,1.081265,8552.215167,15695.091988,0.000000,255.924696,0.959289
2000-07-31,275.634907,0.005471,0.214835,2.074423,8945.476526,6188.434963,0.000000,253.865719,0.515886
2000-08-31,273.643363,0.016486,0.171795,1.208544,8787.250911,10114.130241,0.000000,258.913569,0.355015
2000-09-30,266.780844,0.040531,0.148384,0.857759,7687.016192,12888.810599,0.000000,265.094486,0.186357
2000-10-31,259.412862,0.029951,0.113129,0.231264,5685.865842,19583.623736,0.551964,273.551964,0.138155


In [4]:
#train baseline prediction regression model only fitted on train data
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
# lag_scaler = StandardScaler()


x_lagged = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)

x_train = x_lagged[:'2017-12-31']
x_val = x_lagged['2018-01-01':'2021-12-31']
x_test = x_lagged['2022-01-01':]

y_train = x_lagged[:'2017-12-31']
y_val = x_lagged['2018-01-01':'2021-12-31']
y_test = x_lagged['2022-01-01':]

X_train_lagged = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_train_lagged = y_train[['runoff']]
Y_val_lagged = y_val[['runoff']]
X_val_lagged = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
X_test_lagged = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_test_lagged = x_test[['runoff']]

X_train_val = pd.concat([X_train_lagged, X_val_lagged])
Y_train_val = pd.concat([Y_train_lagged, Y_val_lagged])

# lag_scaler.fit(X_train_lagged)
 
# x_train_scaled = lag_scaler.transform(X_train_lagged)
# x_val_scaled = lag_scaler.transform(X_val_lagged)
# x_test_scaled = lag_scaler.transform(X_test_lagged)

model = LinearRegression()
model.fit(X_train_lagged, Y_train_lagged)

y_test_pred = model.predict(X_test_lagged)
baseline_nse = calculate_nse(Y_test_lagged, y_test_pred)
print(baseline_nse)
# alpha, beta, gamma, delta=amodel.coef_
print(model.coef_)

runoff    0.412207
dtype: float64
[[ 3.01059618e-01  7.78374296e-02 -2.00126628e-06  1.43705861e+01]]


/Users/abhimanyu/Desktop/projects/the_indus_project/gisenv/lib/python3.13/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


In [5]:
train_preds = model.predict(X_train_lagged)
val_preds = model.predict(X_val_lagged)
test_preds = model.predict(X_test_lagged)

#recombination
baseline_q_no_leak = pd.concat([
    pd.DataFrame(train_preds, index=X_train_lagged.index, columns=['baseline_q']),
    pd.DataFrame(val_preds, index=X_val_lagged.index, columns=['baseline_q']),
    pd.DataFrame(test_preds, index=X_test_lagged.index, columns=['baseline_q'])
])
Y_full = pd.concat([Y_train_val, Y_test_lagged])

df_full = pd.concat([X_train_lagged, X_val_lagged, X_test_lagged])
df_full['baseline_q'] = baseline_q_no_leak
df_full['residuals'] = Y_full['runoff'] - df_full['baseline_q']

df_adv = create_lag_adv_corrected(df1) 
df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()

In [6]:
from xgboost import XGBRegressor
import pandas as pd

xgb = XGBRegressor(
    max_depth=5,
    learning_rate=0.01,
    n_estimators=1500,
    random_state=42, # Add for reproducibility
)

train_ml = df_adv[:'2017-12-31']
val_ml = df_adv['2018-01-01':'2021-12-31']
test_ml = df_adv['2022-01-01':]

X_train_ml = train_ml.drop(columns=['baseline_q', 'residuals'])
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(columns=['baseline_q', 'residuals'])
y_val_ml = val_ml['residuals']

# 4. Fit the model ONLY on the training data
xgb.fit(X_train_ml, y_train_ml)

# 5. Predict on the validation set
residual_pred_val = xgb.predict(X_val_ml)
q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# 6. Evaluate against the correct validation ground truth
y_val_true = Y_full['runoff'].loc[val_ml.index]
hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

print(f"Hybrid NSE (Validation): {hybrid_nse_val:.4f}")


Hybrid NSE (Validation): 0.8720


In [7]:
# --- Final Test Set Evaluation --- train model on train + val data

# Prepare combined training/validation data
X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, y_val_ml])

# Prepare test data
X_test_ml = test_ml.drop(columns=['baseline_q', 'residuals'])

# Re-fit the model on all available training data
xgb.fit(X_train_val_ml, y_train_val_ml)

# Predict on the test set
residual_pred_test = xgb.predict(X_test_ml)
q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# Evaluate against the test ground truth
y_test_true = Y_full['runoff'].loc[test_ml.index]
hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

Final Hybrid NSE (Test): 0.8932


In [11]:

df_adv = create_lag_adv_corrected(df1) 
df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()
df_adv.head()

col_names = df_adv.columns.tolist()
col_names

['dewpoint',
 'et_loss',
 'ndvi',
 'precipitation',
 'radiation',
 'sca',
 'dd',
 'temp',
 'runoff',
 'sca_lagged',
 'dd_lagged',
 'et_loss_lagged',
 'melt_proxy',
 'precipitation_lagged',
 'precipitation_lagged_cum',
 'sca_lagged_1',
 'sca_lagged_2',
 'sca_lagged_3',
 'sca_lagged_4',
 'sca_lagged_5',
 'sca_lagged_6',
 'sca_lagged_7',
 'dd_lagged_1',
 'dd_lagged_2',
 'dd_lagged_3',
 'dd_lagged_4',
 'dd_lagged_5',
 'dd_lagged_6',
 'dd_lagged_7',
 'et_loss_lagged_1',
 'precipitation_lagged_1',
 'melt_proxy_1_1',
 'melt_proxy_1_2',
 'melt_proxy_2_1',
 'melt_proxy_2_2',
 'melt_proxy_1_3',
 'melt_proxy_1_4',
 'melt_proxy_1_5',
 'melt_proxy_1_6',
 'melt_proxy_1_7',
 'melt_proxy_2_3',
 'melt_proxy_2_4',
 'melt_proxy_2_5',
 'melt_proxy_2_6',
 'melt_proxy_2_7',
 'melt_proxy_3_1',
 'melt_proxy_3_2',
 'melt_proxy_3_3',
 'melt_proxy_3_4',
 'melt_proxy_3_5',
 'melt_proxy_3_6',
 'melt_proxy_3_7',
 'melt_proxy_4_1',
 'melt_proxy_4_2',
 'melt_proxy_4_3',
 'melt_proxy_4_4',
 'melt_proxy_4_5',
 'melt_pr

In [13]:

df_adv = create_lag_adv_corrected(df1) 
df_adv = df_adv.join(df_full[['baseline_q', 'residuals']]).dropna()
from xgboost import XGBRegressor
import pandas as pd

# --- FIX: Explicitly define the feature set to exclude current-day data ---
# These are the features created by create_lag_adv_corrected
lagged_feature_columns = [
    'sca_lagged_1', 'sca_lagged_2',
    'dd_lagged_1', 'dd_lagged_2',
    'et_loss_lagged_1',
    'precipitation_lagged_1',
    'melt_proxy_1_1', 'melt_proxy_1_2',
    'melt_proxy_2_1', 'melt_proxy_2_2'
]

# These are the columns we want to predict or use in the final calculation
target_and_baseline_cols = ['baseline_q', 'residuals']

# The final dataframe for the ML model should ONLY contain these columns
df_ml_final = df_adv[lagged_feature_columns+ target_and_baseline_cols]


# --- Now proceed with the corrected dataframe ---
xgb1 = XGBRegressor(
    max_depth=5,
    learning_rate=0.01,
    n_estimators=2500,
    random_state=42
)

# 1. Split the clean ML dataframe
train_ml = df_ml_final[:'2017-12-31']
val_ml = df_ml_final['2018-01-01':'2021-12-31']
test_ml = df_ml_final['2022-01-01':]

# 2. Prepare features and labels for validation
X_train_ml = train_ml.drop(columns=target_and_baseline_cols)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(columns=target_and_baseline_cols)

# 3. Fit the model ONLY on the training data
xgb1.fit(X_train_ml, y_train_ml)

# 4. Predict on the validation set
residual_pred_val = xgb1.predict(X_val_ml)
q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# 5. Evaluate against the correct validation ground truth
y_val_true = Y_full['runoff'].loc[val_ml.index]
hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# --- Final Test Set Evaluation ---
# Prepare combined training/validation data
X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, val_ml['residuals']])

# Prepare test data
X_test_ml = test_ml.drop(columns=target_and_baseline_cols)

# Re-fit the model on all available training data
xgb1.fit(X_train_val_ml, y_train_val_ml)

# Predict on the test set
residual_pred_test = xgb1.predict(X_test_ml)
q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# Evaluate against the test ground truth
y_test_true = Y_full['runoff'].loc[test_ml.index]
hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

Corrected Hybrid NSE (Validation): 0.3880
Final Hybrid NSE (Test): 0.4249


In [22]:

from xgboost import XGBRegressor
import pandas as pd

# --- FIX: Explicitly define the feature set to exclude current-day data ---
# These are the features created by create_lag_adv_corrected
lagged_feature_columns = [
    'sca_lagged_1',
 'sca_lagged_2',
 'sca_lagged_3',
 'sca_lagged_4',
 'sca_lagged_5',
 'sca_lagged_6',
 'sca_lagged_7',
 'dd_lagged_1',
 'dd_lagged_2',
 'dd_lagged_3',
 'dd_lagged_4',
 'dd_lagged_5',
 'dd_lagged_6',
 'dd_lagged_7',
 'et_loss_lagged_1',
 'precipitation_lagged_1',
 'melt_proxy_1_1',
 'melt_proxy_1_2',
 'melt_proxy_2_1',
 'melt_proxy_2_2',
 'melt_proxy_1_3',
 'melt_proxy_1_4',
 'melt_proxy_1_5',
 'melt_proxy_1_6',
 'melt_proxy_1_7',
 'melt_proxy_2_3',
 'melt_proxy_2_4',
 'melt_proxy_2_5',
 'melt_proxy_2_6',
 'melt_proxy_2_7',
 'melt_proxy_3_1',
 'melt_proxy_3_2',
 'melt_proxy_3_3',
 'melt_proxy_3_4',
 'melt_proxy_3_5',
 'melt_proxy_3_6',
 'melt_proxy_3_7',
 'melt_proxy_4_1',
 'melt_proxy_4_2',
 'melt_proxy_4_3',
 'melt_proxy_4_4',
 'melt_proxy_4_5',
 'melt_proxy_4_6',
 'melt_proxy_4_7',
 'melt_proxy_5_1',
 'melt_proxy_5_2',
 'melt_proxy_5_3',
 'melt_proxy_5_4',
 'melt_proxy_5_5',
 'melt_proxy_5_6',
 'melt_proxy_5_7',
 'melt_proxy_6_1',
 'melt_proxy_6_2',
 'melt_proxy_6_3',
 'melt_proxy_6_4',
 'melt_proxy_6_5',
 'melt_proxy_6_6',
 'melt_proxy_6_7',
 'melt_proxy_7_1',
 'melt_proxy_7_2',
 'melt_proxy_7_3',
 'melt_proxy_7_4',
 'melt_proxy_7_5',
 'melt_proxy_7_6',
 'melt_proxy_7_7']

# These are the columns we want to predict or use in the final calculation
target_and_baseline_cols = ['baseline_q', 'residuals']

# The final dataframe for the ML model should ONLY contain these columns
df_ml_final = df_adv[lagged_feature_columns+ target_and_baseline_cols]


# --- Now proceed with the corrected dataframe ---
xgb1 = XGBRegressor(
    max_depth=5,
    learning_rate=0.01,
    n_estimators=1500,
    random_state=42
)

# 1. Split the clean ML dataframe
train_ml = df_ml_final[:'2017-12-31']
val_ml = df_ml_final['2018-01-01':'2021-12-31']
test_ml = df_ml_final['2022-01-01':]

# 2. Prepare features and labels for validation
X_train_ml = train_ml.drop(columns=target_and_baseline_cols)
y_train_ml = train_ml['residuals']

X_val_ml = val_ml.drop(columns=target_and_baseline_cols)

# 3. Fit the model ONLY on the training data
xgb1.fit(X_train_ml, y_train_ml)

# 4. Predict on the validation set
residual_pred_val = xgb1.predict(X_val_ml)
q_hybrid_val = residual_pred_val + val_ml['baseline_q']

# 5. Evaluate against the correct validation ground truth
y_val_true = Y_full['runoff'].loc[val_ml.index]
hybrid_nse_val = calculate_nse(y_val_true, q_hybrid_val)

print(f"Corrected Hybrid NSE (Validation): {hybrid_nse_val:.4f}")

# --- Final Test Set Evaluation ---
# Prepare combined training/validation data
X_train_val_ml = pd.concat([X_train_ml, X_val_ml])
y_train_val_ml = pd.concat([y_train_ml, val_ml['residuals']])

# Prepare test data
X_test_ml = test_ml.drop(columns=target_and_baseline_cols)

# Re-fit the model on all available training data
xgb1.fit(X_train_val_ml, y_train_val_ml)

# Predict on the test set
residual_pred_test = xgb1.predict(X_test_ml)
q_hybrid_test = residual_pred_test + test_ml['baseline_q']

# Evaluate against the test ground truth
y_test_true = Y_full['runoff'].loc[test_ml.index]
hybrid_nse_test = calculate_nse(y_test_true, q_hybrid_test)

print(f"Final Hybrid NSE (Test): {hybrid_nse_test:.4f}")

Corrected Hybrid NSE (Validation): 0.6417
Final Hybrid NSE (Test): 0.7194
